# T/m relationships per dataset as a function of `BASE_PERIOD`

**T** = number of residual observations available for covariance estimation

**m** = the height of the summing matrix > BASE_PERIOD + 1

$$T = \frac{\text{train\_days}}{\text{BASE\_PERIOD}},
\qquad \text{train\_days} = L_{\text{trimmed}} - \text{HORIZON\_RATIO} \cdot \text{BASE\_PERIOD}$$


$$\text{BASE\_PERIOD} \propto m$$

$$\text{BASE\_PERIOD} \propto \frac{1}{T}$$

$$ \frac{T}{m} \propto \frac{1}{\text{BASE\_PERIOD}^2}$$

Conclusion: A BASE_PERIOD <= 14 is the safest bet, for all datasets, allowing several intermediate levels to still fulfill the T > 3m rule of thumb for covariance shrinkage estimation

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..')  # Run from repo root

import numpy as np
import pandas as pd
import plotly.express as px

import python.utils as utils

## Settings

In [ ]:
BASE_PERIODS = [7, 14, 28, 56, 84, 112]
HORIZON_RATIO = 1 # Cut off from train data
N_SERIES = None
DATASETS = ['web-traffic', 'M5', 'M4']

In [ ]:
def raw_series_lengths(dataset: str, n_series: int | None) -> np.ndarray:
    '''Number of usable daily observations per series after ffill, before BASE_PERIOD
    trimming. ffill makes the span contiguous, so the usable length is
    (last_valid - first_valid_after_gaps). We approximate it by the count of days
    from first to last valid observation (inclusive), which is what ffill yields.'''
    if dataset == 'web-traffic':
        X = utils.fetch_dataset_web_traffic()
        X = utils.filter_last_days_available(X, 28 * 100)
    elif dataset == 'M5':
        X = utils.fetch_dataset_m5()
    elif dataset == 'M4':
        X = utils.fetch_dataset_m4()
    else:
        raise ValueError(dataset)

    if utils.is_long(X):
        X = utils.long_to_wide(X)

    if n_series is not None:
        X = X.iloc[:, :n_series]

    first = X.apply(lambda c: c.first_valid_index())
    last = X.apply(lambda c: c.last_valid_index())
    span_days = (last - first).dt.days + 1
    return span_days.dropna().astype(int).to_numpy()


raw_lengths = {ds: raw_series_lengths(ds, N_SERIES) for ds in DATASETS}
{ds: (len(v), int(v.min()), int(v.max())) for ds, v in raw_lengths.items()}

In [ ]:
def T_for(lengths: np.ndarray, base_period: int, horizon_ratio: int) -> np.ndarray:
    horizon_days = base_period * horizon_ratio
    trimmed = (lengths // base_period) * base_period
    train_days = trimmed - horizon_days
    T = np.floor(train_days / base_period).astype(int)
    return np.clip(T, 0, None)


rows = []
for ds in DATASETS:
    L = raw_lengths[ds]
    for P in BASE_PERIODS:
        T = T_for(L, P, HORIZON_RATIO)
        usable = T[T > 0]
        n_dropped = int((T <= 0).sum())
        rows.append({
            'dataset': ds,
            'BASE_PERIOD': P,
            'm_min': P + 1,
            'n_series': len(T),
            'n_dropped': n_dropped,
            'T_min': int(usable.min()) if usable.size else 0,
            'T_mean': float(usable.mean()) if usable.size else 0.0,
            'T_max': int(usable.max()) if usable.size else 0,
            'T_over_m_mean': float(usable.mean()) / (P + 1) if usable.size else 0.0,
        })

T_table = pd.DataFrame(rows)
T_table

## Plots

In [ ]:
plot_df = T_table.sort_values(['dataset', 'BASE_PERIOD'])
fig = px.line(
    plot_df,
    x='BASE_PERIOD',
    y='T_over_m_mean',
    color='dataset',
    markers=True,
    log_y=True,
    labels={
        'BASE_PERIOD': 'BASE_PERIOD (days)',
        'T_over_m_mean': 'mean T / m  (m = BASE_PERIOD)',
        'dataset': 'dataset',
    },
    title=f'Covariance-estimation regime: mean T/m vs BASE_PERIOD  —  HORIZON_RATIO={HORIZON_RATIO}',
)
fig.add_hline(y=3.0, line_dash='dot', line_color='gray', annotation_text='T = 3m (rule of thumb)')
fig.update_layout(width=800, height=500)
fig.show()

In [ ]:
RATIO = 3  # target T / m

mmax_rows = []
for ds in DATASETS:
    L = raw_lengths[ds]
    for P in BASE_PERIODS:
        T = T_for(L, P, HORIZON_RATIO)
        usable = T[T > 0]
        if not usable.size:
            continue
        t_min, t_mean, t_max = usable.min(), usable.mean(), usable.max()
        mmax_rows.append({
            'dataset': ds,
            'BASE_PERIOD': P,
            'm_min': P + 1,                       # mandatory bottom-level dimension
            'm_max_min_T': t_min / RATIO,        # worst-case (shortest) series
            'm_max_mean_T': t_mean / RATIO,      # typical series
            'm_max_max_T': t_max / RATIO,        # best-case (longest) series
            # Extra dimension affordable beyond the bottom level (typical series):
            'headroom_over_bottom': t_mean / RATIO - P,
        })

mmax_table = pd.DataFrame(mmax_rows)
mmax_table

In [ ]:
import plotly.graph_objects as go

plot_df = mmax_table.sort_values(['dataset', 'BASE_PERIOD'])
fig = px.line(
    plot_df,
    x='BASE_PERIOD',
    y='m_max_mean_T',
    color='dataset',
    markers=True,
    log_y=True,
    labels={
        'BASE_PERIOD': 'BASE_PERIOD (days)',
        'm_max_mean_T': 'max affordable m  (= T / 3)',
        'dataset': 'dataset',
    },
    title=f'Largest hierarchy dimension supporting T = 3m  —  HORIZON_RATIO={HORIZON_RATIO}',
)
# Rename the mean traces and attach a shaded min/max band per dataset (matching colors).
for tr in fig.data:
    ds = tr.name
    tr.name = f'{ds} (m_max, typical T)'
    sub = plot_df[plot_df['dataset'] == ds].sort_values('BASE_PERIOD')
    if sub.empty:
        continue

# Reference: the mandatory bottom-level dimension m_min = BASE_PERIOD + 1.
bp = sorted(BASE_PERIODS)
fig.add_trace(go.Scatter(
    x=bp,
    y=bp,
    mode='lines',
    line=dict(color='black', dash='dash', width=1.5),
    name='m_min = BASE_PERIOD + 1 (mandatory)',
))
fig.update_layout(width=800, height=500)
fig.show()